# [Anthropic API](https://platform.claude.com/docs/en/home)

In [ ]:
from anthropic import Anthropic
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"
max_tokens = 100

In [ ]:
message = client.messages.create(
    model=model,
    max_tokens=max_tokens,
    messages=[{"role": "user", "content": "Hello Claude, how are you?"}],
)

# Display the main content
print(f"Response: {message.content[0].text}")
print(f"Model: {message.model}")
print(f"Stop Reason: {message.stop_reason}")
print(f"Role: {message.role}")
print("Usage:")
print(f"  Input tokens: {message.usage.input_tokens}")
print(f"  Output tokens: {message.usage.output_tokens}")

## [In-Memory Math-tutor Chat Agent](https://platform.claude.com/docs/en/build-with-claude/working-with-messages)

### Selection Probability

Before generating each token (word), the model calculates a probability for every possible next word based on the context:

```
Input: "What do you think"
Next token probabilities:
  about  → 30%
  would  → 20%
  of     → 10%
  is     → 10%
  when   → 10%
  makes  → 5%
  we     → 5%
```

The model then **selects one token** based on these probabilities. A higher probability = more likely to be chosen.

### Temperature: Controlling Randomness

**Temperature** is a parameter that adjusts these selection probabilities. Using our example above:

| Range | Level | Effect on Example | Use Cases |
|---|---|---|---|
| **0.0 - 0.3** | **Low** | `about → 100%`, others → 0% | Factual responses, coding, data extraction, content moderation |
| **0.4 - 0.7** | **Medium** | `about → 30%`, `would → 20%`, `of → 10%` (unchanged) | Summarization, educational content, problem-solving, creative writing with constraints |
| **0.8 - 1.0+** | **High** | `about → 23%`, `would → 18%`, `of → 15%` (flattened distribution) | Brainstorming, creative writing, marketing, joke generation |

**Key principle:** Use lower temperatures when accuracy and consistency matter; use higher temperatures when diversity and creativity matter.

**In this tutor:** `temperature=1.0` keeps responses natural and varied without becoming nonsensical.

In [ ]:
def creative_tutor_math(messages: list[dict[str, str]]) -> str:
    """Generate a tutoring response using the math tutor system prompt.

    Args:
        messages: List of message dictionaries with 'role' and 'content' keys.

    Returns:
        str: The assistant's tutoring response.
    """
    system_prompt = """
        You are a patient math tutor.
        Do not directly answer a student's questions.
        Guide them to a solution step by step.
    """
    temperature = 0.0  # Adjust creativity level

    message = client.messages.create(
        model=model,
        max_tokens=200,
        messages=messages,
        system=system_prompt,
        temperature=temperature,
    )
    return message.content[0].text


messages: list[dict[str, str]] = []

# Use a 'while True' loop to run the chatbot forever, press ESC to stop
while True:
    try:
        # Get user input
        user_input = input("> ")
        print(">", user_input)

        # Add user input to the list of messages
        messages.append({"role": "user", "content": user_input})

        # Call Claude with the 'creative_tutor_math' function
        response = creative_tutor_math(messages)
        display(Markdown(response))

        # Add generated text to the list of messages
        messages.append({"role": "assistant", "content": response})
    except Exception as e:
        if "400" in str(e):
            break
        raise

## [Streaming Messages](https://platform.claude.com/docs/en/build-with-claude/streaming)

When streaming is enabled, the API sends multiple events instead of a single response. Each event type represents a step in the message generation:

| Event | Description |
|-------|-------------|
| **message_start** | Contains a Message object with empty content. Emitted once at the start of streaming. |
| **content_block_start** | Marks the beginning of a new content block (text, tool use, etc.). |
| **content_block_delta** | Contains a chunk of generated text or tool data for the current content block. Multiple deltas are sent as the block is generated. |
| **content_block_delta** |  |
| **content_block_stop** | Marks the end of the current content block. |
| **message_delta** | Contains top-level changes to the Message (e.g., final stop reason, token usage updates). |
| **message_stop** | Marks the end of the message stream. The complete message is now available. |

In [ ]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "Hello Claude, how are you? What is the weather like today?"}
    ],
    temperature=1.0,
) as stream:
    for text in stream.text_stream:
        print(text, end="")

## [Structured data](https://platform.claude.com/docs/en/test-and-evaluate/develop-tests#grade-your-evaluationsz)

In [ ]:
import json

# Structured data

response = client.messages.create(
    model=model,
    max_tokens=200,
    messages=[
        {"content": "Generate a random user data in json format", "role": "user"},
        {
            "content": "Here is it, just one user, with fields: user_id, name, email, and phone. Nothing more...\n```json",
            "role": "assistant",
        },
    ],
    stop_sequences=["```"],
)

print("AI message:", response.content[0].text)

event_bridge_rule = json.loads(response.content[0].text)
print("Json parsed:", event_bridge_rule)

## [Prompt evaluation](https://platform.claude.com/docs/en/test-and-evaluate/develop-tests#grade-your-evaluations) & [Prompt engineering](https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/overview)

Graders:
- Code
- Model
- Human


In [99]:
python_criteria = """
The function must satisfy ALL of the following:
    - Uses a single list comprehension with a single compound condition (no nested comprehensions or helper functions).
    - Accesses the state via `instance.get('State') == 'running'` (flat string value, NOT a nested dict like `instance.get('State', {}).get('Name')`).
    - Uses `instance.get('InstanceType', '').startswith(('t2', 't3'))` with a TUPLE argument (not two separate startswith calls).
    - Includes proper type hints: parameter annotated as `list[dict[str, Any]]`, return type as `list[dict[str, Any]]`.
    - Imports `Any` from the `typing` module.
    - Function name must be short but meaningful.
"""

json_criteria = """
The JSON policy must satisfy ALL of the following:
    1. Contains exactly THREE Statement entries.
    2. Statement 1 (Sid=AllowGetObject): Effect Allow, s3:GetObject, IpAddress condition 192.168.1.0/24.
    3. Statement 2 (Sid=DenyGetObjectOtherIPs): Effect Deny, s3:GetObject, NotIpAddress condition 192.168.1.0/24.
    4. Statement 3 (Sid=DenyInsecureTransport): Effect Deny, s3:GetObject, Condition Bool aws:SecureTransport=false — enforces HTTPS.
    5. Principal in all statements must be in object form {AWS: *}, NOT bare string *.
    6. Resource in all statements must be a JSON array containing both the bucket ARN (arn:aws:s3:::my-secure-bucket) AND the objects ARN (arn:aws:s3:::my-secure-bucket/*).
    7. Uses Version 2012-10-17.
"""

test_cases = [
    {
        "task": """
        Write a Python function that takes a list of EC2 instance dictionaries
        and returns only the instances where the state is 'running' and the instance type starts with 't2' or 't3'.
    """,
        "structured_output": "The output must be a pure Python function definition with no additional text or explanation. And without ``` md output fences.",
        "format": "python",
        "criteria": python_criteria,
    },
    # Prompt engineered version
    {
        "task": """
        Write a Python function that takes a list of EC2 instance dictionaries
        and returns only the instances where the state is 'running' and the instance type starts with 't2' or 't3'.

        Guidance:
        - Include proper type hints: parameter as list[dict[str, Any]],
        - Use a single list comprehension to filter the instances based on the specified conditions.
        - Your output must be a single function definition with docstring or comments.
        - Output must be compileable Python code.

        Follow these steps:
        1. Define the function with the meaningful but short name.
        2. Access the state with instance.get('State').
        3. Check instance type with instance.get('InstanceType').

        Output example:
        ```python
        from typing import Any

        def func(instances: list[dict[str, Any]]) -> list[dict[str, Any]]:
            return [State == 'running' and InstanceType.startswith(('t2', 't3'))]
        ```
    """,
        "structured_output": "The output must be a pure Python function definition with no additional text or explanation. And without ``` md output fences.",
        "format": "python",
        "criteria": python_criteria,
    },
    #   {
    #     "task": f"""
    #         Write a JSON object representing an S3 bucket policy that allows only GetObject actions
    #         from a specific IP address range (192.168.1.0/24) on all objects within the bucket 'my-secure-bucket'.
    #     """,
    #     "structured_output": "The output must be a valid JSON object with no additional text or explanation. And without ``` json output fences.",
    #     "format": "json",
    #     "criteria": json_criteria,
    #   }
]


def grade_json(text: str) -> int:
    try:
        json.loads(text)
        return 10
    except json.JSONDecodeError:
        return 0


def grade_python_code(text: str) -> int:
    try:
        compile(text, "<string>", "exec")
        return 10
    except SyntaxError:
        return 0


def model_grading_function(response: str, test_case: dict) -> str:
    eval_prompt = f"""
You are an expert code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{response}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "score": A number between 1-10
- "reasoning": A concise explanation of your overall assessment, 10 tokens or less

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "score": number,
    "reasoning": string
}}
    """

    grade_response = client.messages.create(
        model=model,
        max_tokens=100,
        messages=[
            {
                "content": f"Evaluate the following response for the task: {test_case['task']}\nResponse: {response}\nUse the criteria: {test_case['criteria']}",
                "role": "user",
            },
            {"content": "```json", "role": "assistant"},
        ],
        stop_sequences=["```"],
        temperature=0.0,
        system=eval_prompt,
    )
    return json.loads(grade_response.content[0].text.strip())


for test_case in test_cases:
    response = client.messages.create(
        model=model,
        max_tokens=300,
        messages=[
            {"content": test_case["task"], "role": "user"},
            {"content": test_case["structured_output"], "role": "user"},
            # {"content": "Here is the solution:\n```", "role": "assistant"},
        ],
    )

    code_score = 0  # between 0-10 | 10 true, 0 false
    if test_case["format"] == "json":
        code_score = grade_json(response.content[0].text)
    elif test_case["format"] == "python":
        code_score = grade_python_code(response.content[0].text)

    model_grade = model_grading_function(response.content[0].text, test_case)

    score = (model_grade["score"] + code_score) / 2

    print(f"Test Case: {test_case['task']}")
    print(f"Code Score: {code_score}")
    print(f"Model Score: {model_grade['score']} - {model_grade['reasoning']}")
    print(f"Score: {score}")
    print(f"Response: \n{response.content[0].text}")
    print("-" * 50)
    print()

Test Case: 
        Write a Python function that takes a list of EC2 instance dictionaries
        and returns only the instances where the state is 'running' and the instance type starts with 't2' or 't3'.
    
Code Score: 0
Model Score: 4 - Missing imports, wrong dict keys ('state'/'instance_type' vs 'State'/'InstanceType'), and incomplete type hints (missing `Any` import and `dict[str, Any]`).
Score: 2.0
Response: 

```python
def filter_ec2_instances(instances: list[dict]) -> list[dict]:
    """
    Filter EC2 instances based on state and instance type.
    
    Args:
        instances: List of EC2 instance dictionaries
        
    Returns:
        List of instances where state is 'running' and instance type starts with 't2' or 't3'
    """
    return [
        instance for instance in instances
        if instance.get('state') == 'running' and 
        instance.get('instance_type', '').startswith(('t2', 't3'))
    ]
```
--------------------------------------------------

Test Case